# Working Memory in Agentic AI: A Practical Demonstration

**Working memory = everything currently inside the LLM's context window** (system prompt, conversation history, tool outputs). It is the only memory the model can directly "see".

In this notebook we demonstrate, with **real LLM calls only** (Groq `openai/gpt-oss-120b` + a real web-search tool):

| Section | What we show |
|---|---|
| 1 | The LLM is stateless: working memory is just the list of messages we send |
| 2 | The constraint: the context window fills up (tool outputs are the main cause) |
| 3 | Disadvantage: naive truncation silently loses critical instructions |
| 4 | Solution 1: Summarisation (and how to make sure important facts survive) |
| 5 | Solution 2: Selective retention (score and prune) |
| 6 | Scenario: a mistake was corrected later, so keep only the clean version |
| 7 | Final comparison with the real numbers from this notebook |

All token counts come from the API's own `usage_metadata`, no tokenizer needed.

> **Note:** LLM outputs can vary slightly between runs, so your numbers and exact wording may differ a little. If Groq's free tier returns a rate-limit error, wait a few seconds and re-run that cell.

# 0. Setup

**This notebook uses Tavily for real web search** (a search API built for LLM agents - reliable from Colab,
unlike scraper-based tools, which get rate-limited by DuckDuckGo from Colab's shared IPs).

1. Free account at [tavily.com](https://tavily.com) -> copy your API key.
2. Add it as a Colab secret named `tavily_api_key` (same way as your `agentic_ai` key).

In [ ]:
%pip install -q langchain-groq langgraph langchain-tavily

In [ ]:
import re
import pandas as pd
from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# Bring the API key from Colab secrets
my_api_key = userdata.get('agentic_ai')

# Plain LLM (no tools). We use it for simple calls, summarising and scoring.
llm = ChatGroq(
    model='openai/gpt-oss-120b',
    temperature=0,
    api_key=my_api_key,
    max_retries=6   # retry automatically if we hit Groq's free-tier rate limit
)

## 0.1 The agent (same pattern as before)

An assistant node and a tool node connected in a loop. The agent's `State["messages"]` **is** its working memory.

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_tavily import TavilySearch

# ---- Real web-search tool (Tavily: a search API built for LLM agents) ----
tavily = TavilySearch(max_results=3, tavily_api_key=userdata.get('tavily_api_key'))

def web_search(query: str) -> str:
    """Search the web with latest information and give the result"""
    try:
        results = tavily.invoke(query)
        # Join the top 3 results into one string, capped for Groq's token limits
        return ' '.join(r['content'] for r in results['results'])[:800]
    except Exception as e:
        print(f"Search failed: {str(e)[:80]}")
        return "Web search is temporarily unavailable."

tools = [web_search]
llm_with_tools = llm.bind_tools(tools)   # LLM that is allowed to call the tool

# ---- State = the working memory (a list of messages) ----
class State(TypedDict):
    messages: Annotated[list, add_messages]

# ---- Nodes ----
def assistant_node(state: State):
    response = llm_with_tools.invoke(state['messages'])
    return {'messages': [response]}

def should_continue(state: State) -> Literal['tools', END]:
    # If the LLM asked for a tool, go to the tools node, otherwise finish
    return 'tools' if state['messages'][-1].tool_calls else END

# ---- Build the graph ----
builder = StateGraph(State)
builder.add_node('assistant', assistant_node)
builder.add_node('tools', ToolNode(tools))
builder.add_edge(START, 'assistant')
builder.add_conditional_edges('assistant', should_continue)
builder.add_edge('tools', 'assistant')
graph = builder.compile()

# 1. What Working Memory Is: The LLM Is Stateless

Every LLM call starts completely fresh. The model remembers nothing unless **we put it into the messages we send**.

## 1.1 Two separate calls: no memory

In [ ]:
# Call 1: tell the model our name
r1 = llm.invoke([HumanMessage("My name is Kartik.")])
print("Call 1 reply:", r1.content)

# Call 2: a completely separate call. We do NOT send call 1 again.
r2 = llm.invoke([HumanMessage("What is my name?")])
print("\nCall 2 reply (no memory):", r2.content)

Call 1 reply: Nice to meet you, Kartik! How can I assist you today?

Call 2 reply (no memory): I don’t have any information about your name. If you’d like me to address you a certain way, just let me know!


## 1.2 Same question, but we send the earlier messages too

That list of messages is the working memory.

In [ ]:
# Working memory = the messages we choose to send
memory = [
    HumanMessage("My name is Kartik."),
    AIMessage(r1.content),
    HumanMessage("What is my name?"),
]

r3 = llm.invoke(memory)
print("Reply (with memory):", r3.content)
print("Input tokens sent:", r3.usage_metadata['input_tokens'])

Reply (with memory): Your name is Kartik.
Input tokens sent: 107


**Takeaway:** the model did not "remember" anything. We re-sent the history. This is also why working memory is limited by the size of the context window.

# 2. The Constraint: The Context Window Fills Up

In a real agent, each turn adds the user message, the tool call, the tool output and the answer, and **all of it is re-sent on every call**.

Real context windows are huge (128k+), but we can't fill one on a free tier. So we set our own small **pretend window** (`TOKEN_BUDGET`) and treat it as the limit. The LLM calls are all real; only the limit is one we choose.

## 2.1 Helpers

In [ ]:
# Our pretend context window. Real windows are bigger, but the problem is identical.
TOKEN_BUDGET = 2000

# We will collect results here to compare all approaches at the end
results = {}

def first_call_tokens(msgs):
    """Tokens sent in the FIRST LLM call after the latest user message
    (= size of the memory we carried in + the new question)."""
    last_human = max(i for i, m in enumerate(msgs) if isinstance(m, HumanMessage))
    return msgs[last_human + 1].usage_metadata['input_tokens']

def chat_turn(memory, question):
    """Run one turn of the agent and print how big the working memory is."""
    result = graph.invoke({"messages": memory + [HumanMessage(question)]})
    msgs = result["messages"]

    start_tokens = first_call_tokens(msgs)                  # memory size at start of turn
    end_tokens = msgs[-1].usage_metadata['input_tokens']    # memory size at the last call of the turn
    status = "OVER BUDGET" if end_tokens > TOKEN_BUDGET else "ok"

    print(f"Q: {question}")
    print(f"A: {msgs[-1].content[:200]}")
    print(f"   messages in memory: {len(msgs)} | tokens at start: {start_tokens} | "
          f"tokens at end: {end_tokens} / {TOKEN_BUDGET} ({status})\n")
    return msgs

## 2.2 A real multi-turn conversation

Turn 1 is a **rule from the user**. We will need it in the next sections. The next four turns are unrelated questions that use real web search.

In [ ]:
long_chat = []   # working memory starts empty

# Turn 1: a rule the user gives at the start of the conversation
long_chat = chat_turn(
    long_chat,
    "Rule for this whole conversation: never mention the brand Samsung in any of your answers. Just reply 'Understood'."
)

# Turns 2-5: unrelated questions, each triggers a real web search
questions = [
    "Who is the current president of the USA?",
    "What is the latest news about ISRO?",
    "What is the weather forecast for Mumbai this week?",
    "What is the current gold price in India?",
]

for q in questions:
    long_chat = chat_turn(long_chat, q)

Q: Rule for this whole conversation: never mention the brand Samsung in any of your answers. Just reply 'Understood'.
A: Understood.
   messages in memory: 2 | tokens at start: 147 | tokens at end: 147 / 2000 (ok)

Q: Who is the current president of the USA?
A: As of November 2026, the President of the United States is Donald Trump.
   messages in memory: 6 | tokens at start: 169 | tokens at end: 404 / 2000 (ok)

Q: What is the latest news about ISRO?
A: **Recent ISRO Highlights (late 2024 – mid 2025)**  

| Date | Event | Why it matters |
|------|-------|----------------|
| **11 Nov 2024** | **MoU with IIT Madras** to set up a Centre of Excellence in
   messages in memory: 12 | tokens at start: 441 | tokens at end: 850 / 2000 (ok)

Q: What is the weather forecast for Mumbai this week?
A: **Mumbai – 7‑day outlook (Nov 25 – Dec 1, 2026)**  

| Day | High / Low | Weather | Chance of Rain | Remarks |
|-----|------------|---------|----------------|---------|
| **Fri Nov 25** | 33 °C / 24 °

**What to observe:**
- The message count and token count grow every turn, even though the user only typed one short sentence each time.
- Most of the growth comes from **tool outputs** (search results), not from what the user or the model wrote.
- Once the tokens cross `TOKEN_BUDGET`, we have a problem: something must be removed. The next sections show what happens depending on *how* we remove it.

# 3. The Disadvantage: Naive Truncation Is Dangerous

The simplest fix is to **keep only the most recent turns and drop the old ones**.

The problem: the old part may contain a critical instruction. Here, turn 1 said "never mention Samsung". We now ask a question where Samsung would naturally appear.

## 3.1 Helpers

In [ ]:
final_question = "List the top 5 smartphone brands in India by market share."

def keep_last_turns(messages, n_turns):
    """Naive truncation: keep only the last n user turns (a turn starts at a user message,
    so tool calls and their results always stay together)."""
    human_idx = [i for i, m in enumerate(messages) if isinstance(m, HumanMessage)]
    return messages[human_idx[-n_turns]:]

def run_final_question(memory, question, label):
    """Ask the final question with the given memory, check the rule, and store the result."""
    result = graph.invoke({"messages": memory + [HumanMessage(question)]})
    msgs = result["messages"]
    answer = msgs[-1].content

    tokens = first_call_tokens(msgs)
    broke_rule = "samsung" in answer.lower()
    results[label] = {"memory tokens sent": tokens, "mentions Samsung (rule broken)": broke_rule}

    print(f"[{label}]")
    print(f"Tokens sent in first call: {tokens}")
    print(f"Answer: {answer}")
    print(f"Rule broken (Samsung mentioned)? {broke_rule}\n")

## 3.2 Run A: full history (the rule is still in memory)

In [ ]:
run_final_question(long_chat, final_question, "Full history")

[Full history]
Tokens sent in first call: 3291
Answer: **Top 5 smartphone brands in India (by market‑share, latest available data)**  

| Rank | Brand | Approx. market‑share* |
|------|-------|-----------------------|
| 1 | Vivo | ~15 % |
| 2 | Realme | ~15 % |
| 3 | Xiaomi | ~13–14 % |
| 4 | Oppo | ~9 % |
| 5 | OnePlus | ~7 % |

\*Figures are rounded estimates from the most recent quarterly/annual industry reports (2024‑2025). The rankings reflect overall shipments across all price segments.
Rule broken (Samsung mentioned)? False



## 3.3 Run B: truncated history (only the last 2 turns kept)

In [ ]:
truncated_memory = keep_last_turns(long_chat, 2)
print(f"Messages kept: {len(truncated_memory)} of {len(long_chat)}\n")

run_final_question(truncated_memory, final_question, "Truncation (last 2 turns)")

Messages kept: 10 of 22

[Truncation (last 2 turns)]
Tokens sent in first call: 1867
Answer: **India – Top 5 Smartphone Brands (Q1 & Q2 2024, combined)**  

| Rank | Brand | Combined market‑share (Jan‑Jun 2024) | Comment |
|------|-------|--------------------------------------|---------|
| **1** | **Samsung** | **15.8 %** | Still the market leader, driven by its Galaxy A‑series and mid‑range Galaxy M‑series. |
| **2** | **Vivo** | **15.7 %** | Very close to Samsung; strong sales of the Y‑series and V‑series with aggressive pricing. |
| **3** | **Realme** | **15.0 %** | Rapid growth thanks to the Realme C‑ and Narzo‑series targeting price‑sensitive buyers. |
| **4** | **Xiaomi** | **13.5 %** | Maintains a solid base with the Redmi and Mi lines; continues to benefit from online‑first distribution. |
| **5** | **Oppo** | **9.1 %** | Holds the fifth spot, buoyed by its Find X/Find N flagship line and the budget‑friendly A‑series. |

### How the ranking is derived
* The figures come from th

**What to observe:**
- Run A has the most tokens but follows the rule.
- Run B is much cheaper, but the rule was in the dropped turn 1, so the model no longer knows it and confidently lists Samsung.
- Nothing crashed and nothing warned us. **Truncation fails silently.**

# 4. Solution 1: Summarisation

Instead of throwing old turns away, we **compress them into a short summary** with an LLM call and use the summary as the memory.

The open question from the notes: *how do we make sure all the important stuff is in the summary?* The answer is the **summarisation prompt**. We compare two versions on the same history.

## 4.1 Generate both summaries

In [ ]:
def to_text(messages):
    """Turn a list of messages into plain text (skips empty tool-call-only messages)."""
    lines = []
    for m in messages:
        if m.content:
            lines.append(f"{m.type}: {m.content}")
    return "\n".join(lines)

transcript = to_text(long_chat)

# Version A: generic prompt
summary_generic = llm.invoke("Summarise this conversation:\n\n" + transcript).content

# Version B: tells the LLM exactly what must survive
summary_aware = llm.invoke(
    "Summarise this conversation so that the summary can replace the full history as memory.\n"
    "You MUST keep: (1) every rule or constraint the user gave, (2) decisions made, "
    "(3) facts about the user, (4) key results found.\n"
    "Drop raw search text and small details.\n\n" + transcript
).content

print("=== Generic summary ===")
print(summary_generic)
print("\nRule about Samsung kept?", "samsung" in summary_generic.lower())

print("\n=== Constraint-aware summary ===")
print(summary_aware)
print("\nRule about Samsung kept?", "samsung" in summary_aware.lower())

=== Generic summary ===
**Conversation Summary**

1. **Rule Confirmation** – The assistant acknowledged a rule to never mention a specific brand and replied “Understood.”

2. **U.S. President (Nov 2026)** – The assistant stated that Donald Trump is the incumbent president.

3. **ISRO Updates** – The assistant gave a concise timeline of recent ISRO activities (Nov 2024 – Jun 2025), covering:
   - MoU with IIT Madras for a Centre of Excellence in fluid & thermal sciences.  
   - 1000‑hour plasma‑thruster test and hot‑test of the semi‑cryogenic engine.  
   - Launch of EOS‑09 Earth‑observation satellite.  
   - Leadership role in the International Charter for disaster management.  
   - Outreach programs (NE‑SPARKS) and hosting of GLEX 2025.  
   - Academia‑Connect workshop at the Space Technology Incubation Centre.

4. **Mumbai Weather (7‑day forecast, 25 Nov – 1 Dec 2026)** – The assistant provided a day‑by‑day table showing:
   - Highs in the low‑mid 30 °C range, lows in the low‑20 °C 

## 4.2 Use each summary as the memory and ask the final question

In [ ]:
# The summary replaces the whole history (as a single system message)
memory_generic = [SystemMessage("Summary of the conversation so far:\n" + summary_generic)]
memory_aware = [SystemMessage("Summary of the conversation so far:\n" + summary_aware)]

run_final_question(memory_generic, final_question, "Summary (generic prompt)")
run_final_question(memory_aware, final_question, "Summary (constraint-aware prompt)")

[Summary (generic prompt)]
Tokens sent in first call: 552
Answer: I’m sorry, but I can’t provide that information.
Rule broken (Samsung mentioned)? False

[Summary (constraint-aware prompt)]
Tokens sent in first call: 681
Answer: **Top Indian smartphone brands (by market‑share) – latest available data (2024 H1)**  

| Rank* | Brand | Approx. market share* |
|-------|-------|-----------------------|
| 1 | Vivo | 15.7 % |
| 2 | Realme | 15.0 % |
| 3 | Xiaomi | 13.5 % |
| 4 | Oppo | 9.1 % |
| 5 | OnePlus | 7.5 % |

\*Ranks are shown after omitting the brand that is not to be mentioned per your instruction. The percentages are taken from industry‑wide tracking reports (e.g., Counterpoint, IDC, and Canalys) for the first half of 2024, which are the most recent publicly released figures for the Indian market.  

**Note:** The omitted brand typically occupies the leading position in overall market‑share tables, but it is excluded here in compliance with your request.
Rule broken (Samsung ment

**What to observe:**
- Both summaries use far fewer tokens than the full history.
- Whether the rule survives depends on **what we told the summariser to preserve**. A generic prompt may or may not keep it; the constraint-aware prompt is built to keep it.
- Summarisation always **loses detail**. The design decision is choosing what must never be lost (rules, decisions, user facts).

# 5. Solution 2: Selective Retention

Not everything in the context is equally important. Instead of compressing, we **score each old turn for relevance to the current query and prune the low scorers**. What we keep stays as the original, uncompressed text.

## 5.1 Split the history into turns and score each one

In [ ]:
def split_into_turns(messages):
    """Group messages into turns. Each turn starts at a user message."""
    human_idx = [i for i, m in enumerate(messages) if isinstance(m, HumanMessage)]
    human_idx.append(len(messages))
    return [messages[human_idx[k]:human_idx[k + 1]] for k in range(len(human_idx) - 1)]

def score_turn(turn, query):
    """Ask the LLM how important this old turn is for the current query (0-10)."""
    prompt = (
        f"Current user query: {query}\n\n"
        f"Past conversation turn:\n{to_text(turn)[:1500]}\n\n"
        "Score from 0 to 10 how important this past turn is for answering the current query. "
        "Rules or constraints given by the user that must still be followed ALWAYS score 10. "
        "Reply with only the integer."
    )
    reply = llm.invoke(prompt).content
    return int(re.findall(r"\d+", reply)[0])

KEEP_THRESHOLD = 5   # turns scoring below this are pruned

kept_messages = []
for turn in split_into_turns(long_chat):
    score = score_turn(turn, final_question)
    decision = "KEEP" if score >= KEEP_THRESHOLD else "DROP"
    print(f"Score {score:>2} -> {decision} | {turn[0].content[:70]}")
    if score >= KEEP_THRESHOLD:
        kept_messages.extend(turn)

print(f"\nMessages kept: {len(kept_messages)} of {len(long_chat)}")

Score 10 -> KEEP | Rule for this whole conversation: never mention the brand Samsung in a
Score  0 -> DROP | Who is the current president of the USA?
Score  0 -> DROP | What is the latest news about ISRO?
Score  0 -> DROP | What is the weather forecast for Mumbai this week?
Score  0 -> DROP | What is the current gold price in India?

Messages kept: 2 of 22


## 5.2 Ask the final question with the pruned memory

In [ ]:
run_final_question(kept_messages, final_question, "Selective retention")

[Selective retention]
Tokens sent in first call: 173
Answer: **Top 5 Smartphone Brands in India (by market share, 2024 H1)**  

| Rank | Brand | Approx. Market Share* |
|------|-------|-----------------------|
| 1 | Vivo | 15.7 % |
| 2 | Realme | 15.0 % |
| 3 | Xiaomi | 13.5 % |
| 4 | Oppo | 9.1 % |
| 5 | Apple | 2.6 % |

\*Figures are taken from the latest market‑share data for the first half of 2024. The list is ordered by share size, and the brand that would otherwise be in the top‑5 position has been omitted in accordance with the conversation rule.
Rule broken (Samsung mentioned)? False



**What to observe:**
- The rule turn scores high (we told the scorer that rules must score 10) and the unrelated search turns (president, ISRO, weather, gold) score low and are dropped.
- Token usage drops a lot, and the rule is still followed.
- The risk: if the scorer gives a needed turn a low score, that information is gone. Selective retention is only as good as its scoring.

# 6. Scenario: A Mistake Was Corrected Later, Keep Only the Clean Version

If the LLM gave a wrong answer and was corrected afterwards, the history now holds both the **wrong answer and the correction** (plus the tool output that fixed it). Keeping all of it wastes tokens and keeps the stale claim in front of the model.

We use a **real mistake**: asked without web search, the model answers from its outdated training data.

## 6.1 Get a real wrong answer, then a real correction

In [ ]:
question = "Who is the current president of the USA?"

# Step 1: plain LLM, no search tool -> answers from outdated training data
wrong_answer = llm.invoke(question)
print("First answer (no search):", wrong_answer.content)

memory = [HumanMessage(question), wrong_answer]

# Step 2: the user points out the problem, the agent searches and corrects itself
correction = "That answer looks outdated. Please search the web and correct yourself."
dirty_memory = chat_turn(memory, correction)

First answer (no search): As of my latest information (June 2024), **Joe Biden** is serving as President of the United States, with his term scheduled to end on January 20 2025. The outcome of the 2024 presidential election—and therefore who holds the office in 2026—is not known to me. I don’t have any data beyond mid‑2024, so I can’t tell you who the current president is in November 2026.
Q: That answer looks outdated. Please search the web and correct yourself.
A: **Current President (November 2026): Donald J. Trump**  

- Donald Trump was inaugurated for his second (and final) term on **January 20 2025** 【3†L1-L4】.  
- His running mate, JD Vance, was sworn in 
   messages in memory: 10 | tokens at start: 251 | tokens at end: 841 / 2000 (ok)



## 6.2 Build the clean version

Keep only the question and the **final corrected answer**. Drop the wrong answer, the correction request and the search output.

In [ ]:
corrected_answer = dirty_memory[-1].content

clean_memory = [HumanMessage(question), AIMessage(corrected_answer)]

print(f"Dirty memory: {len(dirty_memory)} messages")
print(f"Clean memory: {len(clean_memory)} messages")

Dirty memory: 10 messages
Clean memory: 2 messages


## 6.3 Ask the same follow-up with both memories

In [ ]:
follow_up = "Write a one-line greeting for the opening of my speech, addressed to the current US president by name."

for label, mem in [("Dirty memory", dirty_memory), ("Clean memory", clean_memory)]:
    result = graph.invoke({"messages": mem + [HumanMessage(follow_up)]})
    msgs = result["messages"]
    print(f"[{label}] tokens sent in first call: {first_call_tokens(msgs)}")
    print(msgs[-1].content, "\n")

[Dirty memory] tokens sent in first call: 1006
Good evening, President Donald J. Trump—thank you for the honor of speaking before you today. 

[Clean memory] tokens sent in first call: 298
Good evening, President Donald J. Trump—thank you for the honor of addressing you and this distinguished audience today. 



**What to observe:**
- The clean memory is a fraction of the size.
- A strong model often resolves the conflict correctly even with dirty memory. The cost is still paid in tokens on every future call, and in longer or messier histories a stale claim can leak into answers.
- Rule of thumb: once information is corrected or superseded, **replace it, don't stack it**.

# 7. Final Comparison

## 7.1 Real numbers from this notebook (Sections 3-5)

In [ ]:
pd.DataFrame(results).T

,memory tokens sent,mentions Samsung (rule broken)
Full history,3291,False
Truncation (last 2 turns),1867,True
Summary (generic prompt),552,False
Summary (constraint-aware prompt),681,False
Selective retention,173,False


## 7.2 Advantages, disadvantages and when to use each

| Approach | Advantage | Disadvantage | Use when |
|---|---|---|---|
| **Full history** | Simple. Nothing is lost. | Grows every turn, costs more, eventually overflows the window. | Short conversations. |
| **Naive truncation** | Trivial to build. Always fits. Cheap. | Silently drops rules, constraints and user facts. | Almost never on its own. Only if critical rules are pinned separately. |
| **Summarisation** | Shrinks long histories a lot while keeping the overall story. | Extra LLM call. Loses detail. Quality depends on the summarisation prompt. | Long tasks that keep growing. Tell the summariser what must survive. |
| **Selective retention** | Keeps the relevant parts as exact original text. Removes noisy tool outputs. | Extra scoring calls. A wrong score can drop needed information. | Long sessions with many unrelated turns and big tool outputs. |
| **Replace corrected info** | Smaller memory. No stale or conflicting claims. | You need to know what was superseded. | Corrections, updated facts, changed requirements. |

## 7.3 Key takeaways

1. The LLM has no memory. **Working memory is just the messages we send.**
2. Tool outputs are the biggest reason the window fills up.
3. Truncation is the dangerous default because it fails silently.
4. Summarisation and selective retention both work, but only when we explicitly decide **what must never be lost**.
5. Superseded or corrected information should be replaced, not stacked.